In [ ]:
##installa dipendenze sulla destra

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1,2)
qc.x(1)
qc.measure_all()

In [ ]:
sampler = StatevectorSampler()
result = sampler.run([qc], shots=pow(2,10)).result()
print(result[0].data.meas.get_counts())
qc.draw("mpl")

In [ ]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram


In [ ]:
counts = result[0].data.meas.get_counts()
plot_histogram(counts)
plt.show()

In [ ]:
from qiskit.circuit.library import zz_feature_map

In [ ]:
features = [0.2, 0.4, 0.8]
feature_map = zz_feature_map(feature_dimension=len(features))

In [ ]:
encoded = feature_map.assign_parameters(features)
encoded.draw("mpl")

In [ ]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_ibm_runtime.fake_provider import FakeAuckland, FakeWashingtonV2
from qiskit.transpiler import generate_preset_pass_manager


In [ ]:
backend = FakeAuckland()
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))
depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial", # Fixed layout mapped in circuit order
        seed_transpiler=seed, # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

pass_manager.run(ghz).draw("mpl")

# plt.figure(figsize=(8, 6))
# plt.hist(depths, align="left", color="#AC557C")
# plt.xlabel("Depth", fontsize=14)
# plt.ylabel("Counts", fontsize=14)

In [2]:
from qiskit import QuantumCircuit
from qiskit.circuit import QuantumCircuit, Parameter



In [3]:
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0,1)
circuit.ry(0,0,"a")
circuit.measure_all()

In [6]:
from qiskit.circuit import QuantumCircuit, Parameter

# create the parameter
phi = Parameter('phi')
params = [0.1] * circuit.num_parameters
pub = (circuit, params, 1024)
sampler = SamplerV2()
job = sampler.run([pub])
result = job.result()

In [7]:
data = result[0].data
print(f"Databin: {data}\n")
# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=2>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=2>)

The shape of register `meas` is (1024, 1).

The bytes in register `alpha`, shot by shot:
[[0]
 [0]
 [0]
 ...
 [0]
 [0]
 [0]]

Counts: {'00': 506, '11': 518}


In [5]:
from qiskit.circuit.library import efficient_su2
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import SamplerV2


In [ ]:


n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.measure_all()
circuit.draw("mpl")
observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = SamplerV2()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, params)
job = exact_estimator.run([pub],shots=1024)
result = job.result()
counts_ideal = result[0].data.meas.get_counts()
counts_ideal
# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method


In [ ]:
noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
depolarizing_error(cx_depolarizing_prob, 2), ["cx"])

noisy_estimator = SamplerV2(
options=dict(backend_options=dict(noise_model=noise_model)))
job = noisy_estimator.run([pub])
result = job.result()
counts_ideal = result[0].data.meas.get_counts()
counts_ideal

In [14]:
import pennylane as qml
from jax import numpy as np
import jax

In [26]:
dev1 = qml.device("default.qubit", wires=4)

In [27]:
def circuit(params):
    qml.RX(params[0], wires=0)
    qml.RY(params[1], wires=0)
    return qml.expval(qml.PauliZ(0))
params = np.array([0.54, 0.12])
print(circuit(params))

expval(Z(0))
